# Memoizacija

In [1]:
let f x = 3 * x
let g x = f(x + 14)

val f : int -> int = <fun>


val g : int -> int = <fun>


In [2]:
g 0

- : int = 42


In [3]:
let f x = 2 * x

val f : int -> int = <fun>


In [4]:
g 0

- : int = 42


## Spomin kot dodatni argument

In [6]:
let rec stolpi_s_spominom spomin n =
    match List.assoc_opt n spomin with
    | Some s -> (s, spomin)
    | None ->
        let s, spomin =
          match n with
          | 0 -> (1, spomin)
          | n when n < 0 -> (0, spomin)
          | n ->
              let s1, spomin' = stolpi_s_spominom spomin (n - 1) in
              let s2, spomin'' = stolpi_s_spominom spomin' (n - 2) in
              let s3, spomin''' = stolpi_s_spominom spomin'' (n - 3) in
              (s1 + s2 + s3, spomin''')
        in
        (s, (n, s) :: spomin)

val stolpi_s_spominom : (int * int) list -> int -> int * (int * int) list =
  <fun>


In [7]:
stolpi_s_spominom [] 5

- : int * (int * int) list =
(13, [(5, 13); (4, 7); (3, 4); (2, 2); (1, 1); (-2, 0); (-1, 0); (0, 1)])


In [ ]:
fst (stolpi_s_spominom [] 50)

## Zgoščevalne tabele

In [8]:
let spomin = Hashtbl.create 512

val spomin : ('_weak1, '_weak2) Hashtbl.t = <abstr>


In [9]:
let rec stolpi_s_spominom n =
    match Hashtbl.find_opt spomin n with
    | Some s -> s
    | None ->
        let s =
          match n with
          | 0 -> 1
          | n when n < 0 -> 0
          | n -> stolpi_s_spominom (n - 1) + stolpi_s_spominom (n - 2) + stolpi_s_spominom (n - 3)
        in
        Hashtbl.add spomin n s;
        s

val stolpi_s_spominom : int -> int = <fun>


In [10]:
stolpi_s_spominom 50

- : int = 10562230626642


In [11]:
let dodaj_spomin f_brez_spomina =
    let spomin = Hashtbl.create 512 in
    let f_s_spominom x =
      match Hashtbl.find_opt spomin x with
      | None ->
          let y = f_brez_spomina x in
          Hashtbl.add spomin x y;
          y
      | Some y ->
          y
    in
    f_s_spominom

val dodaj_spomin : ('a -> 'b) -> 'a -> 'b = <fun>


In [8]:
let rec stolpi n =
    print_endline ("Računam " ^ string_of_int n);
    match n with
    | 0 -> 1
    | n when n < 0 -> 0
    | n -> stolpi (n - 1) + stolpi (n - 2) + stolpi (n - 3)

val stolpi : int -> int = <fun>


In [20]:
let stolpi = dodaj_spomin stolpi 

val stolpi : int -> int = <fun>


In [24]:
stolpi 5

Računam 5
Računam 2
Računam -1
Računam 0
Računam 1
Računam -2
Računam -1
Računam 0
Računam 3
Računam 0
Računam 1
Računam -2
Računam -1
Računam 0
Računam 2
Računam -1
Računam 0
Računam 1
Računam -2
Računam -1
Računam 0
Računam 4
Računam 1
Računam -2
Računam -1
Računam 0
Računam 2
Računam -1
Računam 0
Računam 1
Računam -2
Računam -1
Računam 0
Računam 3
Računam 0
Računam 1
Računam -2
Računam -1
Računam 0
Računam 2
Računam -1
Računam 0
Računam 1
Računam -2
Računam -1
Računam 0


- : int = 13


## Statični in dinamični klici

In [ ]:
let f x = 3 * x

In [ ]:
let g x = 14 * f x

In [ ]:
let f x = 2 * x

In [ ]:
g 1

## Memoizacija rekurzivnih funkcij

In [ ]:
let odviti_stolpi k n =
    print_endline ("Računam " ^ string_of_int n);
    match n with
    | 0 -> 1
    | n when n < 0 -> 0
    | n -> k (n - 1) + k (n - 2) + k (n - 3)

val odviti_stolpi : (int -> int) -> int -> int = <fun>


In [ ]:
odviti_stolpi (odviti_stolpi (odviti_stolpi (fun n -> 2 * n))) 5

Računam 5
Računam 2
Računam -1
Računam 0
Računam 1
Računam -2
Računam -1
Računam 0
Računam 3
Računam 0
Računam 1
Računam -2
Računam -1
Računam 0
Računam 2
Računam -1
Računam 0
Računam 1
Računam -2
Računam -1
Računam 0
Računam 4
Računam 1
Računam -2
Računam -1
Računam 0
Računam 2
Računam -1
Računam 0
Računam 1
Računam -2
Računam -1
Računam 0
Računam 3
Računam 0
Računam 1
Računam -2
Računam -1
Računam 0
Računam 2
Računam -1
Računam 0
Računam 1
Računam -2
Računam -1
Računam 0


- : int = 13


In [10]:
let dodaj_spomin odviti_f_brez_spomina =
    let spomin = Hashtbl.create 512 in
    let rec f_s_spominom x =
      match Hashtbl.find_opt spomin x with
      | None ->
          let y = odviti_f_brez_spomina f_s_spominom x in
          Hashtbl.add spomin x y;
          y
      | Some y ->
          y
    in
    f_s_spominom

val dodaj_spomin : (('a -> 'b) -> 'a -> 'b) -> 'a -> 'b = <fun>


In [11]:
let stolpi_s_spominom = dodaj_spomin odviti_stolpi

val stolpi_s_spominom : int -> int = <fun>


In [12]:
stolpi_s_spominom 10

Računam 10
Računam 7
Računam 4
Računam 1
Računam -2
Računam -1
Računam 0
Računam 2
Računam 3
Računam 5
Računam 6
Računam 8
Računam 9


- : int = 274
